In [ ]:
#1. Install Dependencies

In [6]:
%pip install aiohttp aiofiles ijson tqdm


Note: you may need to restart the kernel to use updated packages.


In [ ]:
## 2. Download Scryfall Bulk Data
Downloads the full Scryfall card list as JSONL to `scryfall_all_cards.json`


In [12]:
import os
import sys
import json
import gzip
import ijson
import aiohttp
import asyncio
import aiofiles
from tqdm import tqdm

# ==============================================================
# 1. PLATFORM DETECTION & CONFIGURATION
# ==============================================================
IS_WINDOWS = os.name == 'nt'

# The JSON bulk data file remains in the script directory
JSON_FILE = "scryfall_all_cards.json"
SCYFALL_BULK_DATA_URL = "https://api.scryfall.com/bulk-data"

if IS_WINDOWS:
    # Windows Native Google Drive Paths
    OUTPUT_FOLDER = r"G:\My Drive\New Cards\Magic the Gathering"
    CHECK_FOLDER = r"G:\My Drive\Card Database\Magic the Gathering"
else:
    # Ubuntu Paths (Assumes rclone mount at ~/Desktop/GDrive)
    # Note: Using .expanduser ensures the path works regardless of your Ubuntu username
    OUTPUT_FOLDER = os.path.expanduser("/storage/Tera/Card Database/Magic the Gathering")
    CHECK_FOLDER = os.path.expanduser("/storage/Tera/Card Database/Magic the Gathering")

# Performance Tuning
NUM_WORKERS = 5  # Slightly lowered for Linux stability over rclone
MAX_RETRIES = 3


# ==============================
# Async download worker
# ==============================
async def download_worker(session, queue, pbar):
    """Worker task to asynchronously download card images."""
    while True:
        task = await queue.get()
        if task is None:
            queue.task_done()
            break

        name, lang, url, filename = task
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                os.makedirs(os.path.dirname(filename), exist_ok=True)

 
                async with session.get(url, timeout=30) as resp:
                    if resp.status == 200:
                        content = await resp.read()
                        async with aiofiles.open(filename, "wb") as f:
                            await f.write(content)
                        break
                    elif resp.status == 429:  # Rate limited
                        await asyncio.sleep(2 ** attempt)
                    else:
                        # Keep this one silent too — just skip and retry is not needed, so break
                        break
            except Exception as e:
                if attempt == MAX_RETRIES:
                    pass  # Silently drop the failure
                else:
                    await asyncio.sleep(1)

        pbar.update(1)
        queue.task_done()



# ==============================
# Process JSON and queue tasks
# ==============================
async def process_cards(session: aiohttp.ClientSession):
    print("\n[PROCESS] Starting card image processing...", flush=True)

    search_locations = [
        {"path": OUTPUT_FOLDER, "label": "New Cards"},
        {"path": CHECK_FOLDER, "label": "Card Database"}
    ]

    # Pre-check valid locations
    valid_locations = [loc for loc in search_locations if os.path.exists(loc["path"])]

    if not valid_locations and not IS_WINDOWS:
        print(f"[!] WARNING: No Google Drive folders found. Is rclone mounted?")


 
    print(f"[INFO] Scanning for duplicates in: {[loc['label'] for loc in valid_locations]}\n", flush=True)

    queue = asyncio.Queue()
    total_queued = 0

    # We use ijson to stream the large Scryfall file without crashing RAM
    try:
        with open(JSON_FILE, "rb") as f:  # Opened in binary for ijson
            # Scryfall now ships newline-delimited JSONL (gzipped), so parse with multiple_values=True
            for card in ijson.items(f, "", multiple_values=True):
                name = card.get("name", "N/A")
                lang = card.get("lang", "N/A")

                if "image_uris" not in card:
                    continue
 
                url = card["image_uris"].get("large") or card["image_uris"].get("normal")
                if not url:
                    continue
 
                # Cross-platform filename cleaning
                # Linux is lenient, but we keep the Windows cleaning so the G-Drive sync remains valid
                clean_name = name.replace("/", "_").replace(":", "_").replace("?", "").replace("*", "").replace('"', "")
                filename = f"{clean_name}_{lang}.jpg"
 
                already_exists = False
                found_location_label = ""
 
                # Check folders
                for loc in valid_locations:
                    if os.path.exists(os.path.join(loc["path"], filename)):
                        already_exists = True
                        found_location_label = loc["label"]
                        break
 
                if already_exists:
                    # Skip logged for clarity
                    continue
                else:
                    output_path = os.path.join(OUTPUT_FOLDER, filename)
                    await queue.put((name, lang, url, output_path))
                    total_queued += 1
    except FileNotFoundError:
        print(f"[!] Error: {JSON_FILE} not found.")
        return
 
    print(f"\n[SUMMARY] Total cards queued for download: {total_queued}\n", flush=True)
 
    if total_queued == 0:
        print("[DONE] Everything is already up to date!")
        return
 
    # Setup progress bar and workers
    with tqdm(total=total_queued, desc="MTG Sync Progress", unit="card") as pbar:
        # Create pool of worker tasks
        workers = [asyncio.create_task(download_worker(session, queue, pbar)) for _ in range(NUM_WORKERS)]
 
        # Wait for all items in the queue to be processed
        await queue.join()
 
        # Stop workers
        for _ in workers:
            await queue.put(None)
        await asyncio.gather(*workers)


# ==============================
# Scryfall Bulk Data Helpers
# ==============================
async def fetch_bulk_data_uri(session: aiohttp.ClientSession) -> str:
    async with session.get(SCYFALL_BULK_DATA_URL) as resp:
        resp.raise_for_status()
        data = await resp.json()
        all_cards_data = next((item for item in data.get("data", []) if item.get("type") == "all_cards"), None)
        # Scryfall renamed this field to jsonl_download_uri
        return all_cards_data["jsonl_download_uri"]


async def download_json_file(session: aiohttp.ClientSession, uri: str):
    async with session.get(uri) as resp:
        resp.raise_for_status()
        total_size = int(resp.headers.get('content-length', 0))
        gz_file = JSON_FILE + ".gz"
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"Downloading {JSON_FILE}.gz") as pbar:
            async with aiofiles.open(gz_file, 'wb') as f:
                async for chunk in resp.content.iter_chunked(64 * 1024):  # Larger chunks for faster I/O
                    await f.write(chunk)
                    pbar.update(len(chunk))

    # Decompress to the plain JSONL file the rest of the script reads
    with gzip.open(gz_file, 'rb') as fin, open(JSON_FILE, 'wb') as fout:
        fout.write(fin.read())
    os.remove(gz_file)


async def main_async():
    # Use a custom timeout for the whole session
    timeout = aiohttp.ClientTimeout(total=None, connect=60, sock_read=60)
    async with aiohttp.ClientSession(timeout=timeout) as session:

        if not os.path.exists(JSON_FILE):
            print(f"[JSON] Fetching new Scryfall bulk file...", flush=True)
            uri = await fetch_bulk_data_uri(session)
            await download_json_file(session, uri)
 
        await process_cards(session)


# ==============================
# Jupyter entry point — no asyncio.run(), no sys.exit()
# ==============================
await main_async()


[JSON] Fetching new Scryfall bulk file...



[PROCESS] Starting card image processing...
[!] WARNING: No Google Drive folders found. Is rclone mounted?
[INFO] Scanning for duplicates in: []



<frozen posixpath>:82: RuntimeWarning: coroutine 'main_async' was never awaited



[SUMMARY] Total cards queued for download: 533708



MTG Sync Progress:   0%|          | 22/533708 [00:00<6:04:00, 24.44card/s] 

...

[OK] Ugin's Construct (pt)
[OK] Magnify (it)
[OK] Painful Truths (zhs)
[OK] Ana Disciple (pt)
[OK] Aluren (en)
[OK] Spark of Creativity (fr)
[OK] Healing Hands (zht)
[OK] Shepherding Spirits (de)
[OK] O'aka, Traveling Merchant (ja)
[OK] Putrid Imp (zhs)
[OK] From Beyond (it)
[OK] Wood Sage (ko)
[OK] Inspiring Refrain (es)
[OK] Juniper Order Ranger (zhs)


MTG Sync Progress:   1%|▏         | 7740/533708 [01:15<1:25:43, 102.26card/s]


[OK] Rhystic Study (en)
[OK] Sacred Ground (zhs)
[OK] Chandra, the Firebrand (ko)
[OK] Kolaghan's Command (de)
[OK] Snapping Drake (en)
[OK] Smash (it)
[OK] Locust Miser (de)
[OK] Manaform Hellkite (de)
[OK] Corrupted Crossroads (en)
[OK] Army Ants (ko)
[OK] Zagoth Mamba (es)
[OK] Blaze (de)
[OK] Plains (es)
[OK] Highland Lake (pt)


CancelledError: 

[OK] Sigarda's Summons (es)
[OK] Liquimetal Torque (ko)
[OK] Everflowing Chalice (es)
[ERR] Final failure for Master of Predicaments: Session is closed
[ERR] Final failure for Tawnos, Solemn Survivor: Session is closed
[ERR] Final failure for Spike Drone: Session is closed
[ERR] Final failure for Manta Riders: Session is closed
[ERR] Final failure for Old-Growth Troll: Session is closed
[ERR] Final failure for Shinechaser: Session is closed
[ERR] Final failure for Fake Your Own Death: Session is closed
[ERR] Final failure for Lier, Disciple of the Drowned: Session is closed
[ERR] Final failure for Silvergill Adept: Session is closed
[ERR] Final failure for Xerex Strobe-Knight: Session is closed
[ERR] Final failure for Radioactive Spider: Session is closed
[ERR] Final failure for Winter Moon: Session is closed
[ERR] Final failure for Frenzied Tilling: Session is closed
[ERR] Final failure for Riku of Many Paths: Session is closed
[ERR] Final failure for Serra Redeemer: Session is closed

In [ ]:
## 2. Download Card Images — Organized by Set
Saves every card image into `/storage/Tera/Card Database/Organized Cards/<Set Name>/`


In [8]:
import os
import ijson
import aiohttp
import asyncio
import aiofiles
from collections import Counter
from tqdm import tqdm

# ==============================================================
# Organized-by-Set downloader — per-worker sessions
# ==============================================================
JSON_FILE = "scryfall_all_cards.json"
ORGANIZED_ROOT = os.path.expanduser("/storage/Tera/Full Card Database/New Cards/Magic the Gathering")

NUM_WORKERS = 5
MAX_RETRIES = 5
set_counts = Counter()


# ==============================
# Async download worker — one session per worker
# ==============================
async def download_worker(queue, pbar):
    """Each worker owns its own session so one failure can't kill the pool."""
    timeout = aiohttp.ClientTimeout(total=60, connect=30, sock_read=60)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        while True:
            task = await queue.get()
            if task is None:
                queue.task_done()
                break

            name, set_name, lang, url, filename = task
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    os.makedirs(os.path.dirname(filename), exist_ok=True)

                    async with session.get(url) as resp:
                        if resp.status == 200:
                            content = await resp.read()
                            async with aiofiles.open(filename, "wb") as f:
                                await f.write(content)
                            set_counts[set_name] += 1
                            break
                        elif resp.status == 429:  # rate limited — back off and retry
                            await asyncio.sleep(2 ** attempt)
                        else:
                            break  # permanent error (404 etc.) — skip silently
                except (aiohttp.ClientError, asyncio.TimeoutError):
                    if attempt < MAX_RETRIES:
                        await asyncio.sleep(attempt)  # small backoff between retries

            pbar.update(1)
            queue.task_done()


# ==============================
# Process JSONL, grouping cards by set
# ==============================
async def process_cards():
    print("\n[PROCESS] Scanning cards and grouping by set...", flush=True)

    queue = asyncio.Queue()
    total_queued = 0

    try:
        with open(JSON_FILE, "rb") as f:
            for card in ijson.items(f, "", multiple_values=True):
                name = card.get("name", "N/A")
                lang = card.get("lang", "N/A")
                set_name = card.get("set_name", "Unknown Set")

                if "image_uris" not in card:
                    continue

                url = card["image_uris"].get("large") or card["image_uris"].get("normal")
                if not url:
                    continue

                clean_set = set_name.replace("/", "_").replace(":", "_").replace("?", "").replace("*", "").replace('"', "")
                clean_name = name.replace("/", "_").replace(":", "_").replace("?", "").replace("*", "").replace('"', "")
                filename = f"{clean_name}_{lang}.jpg"
                output_path = os.path.join(ORGANIZED_ROOT, clean_set, filename)

                if os.path.exists(output_path):
                    continue

                await queue.put((name, set_name, lang, url, output_path))
                total_queued += 1
    except FileNotFoundError:
        print(f"[!] Error: {JSON_FILE} not found.")
        return

    print(f"\n[SUMMARY] Total cards queued for download: {total_queued}\n", flush=True)

    if total_queued == 0:
        print("[DONE] Everything is already downloaded!")
        return

    with tqdm(total=total_queued, desc="Organized Sync", unit="card") as pbar:
        workers = [asyncio.create_task(download_worker(queue, pbar)) for _ in range(NUM_WORKERS)]

        await queue.join()

        for _ in workers:
            await queue.put(None)
        await asyncio.gather(*workers)

    print("\n[SET SUMMARY] Cards downloaded by set:")
    for set_name, count in sorted(set_counts.items()):
        print(f"  {set_name}: {count}")


# ==============================
# Main
# ==============================
async def main_async():
    await process_cards()


await main_async()



[PROCESS] Scanning cards and grouping by set...

[SUMMARY] Total cards queued for download: 0

[DONE] Everything is already downloaded!
